<center><p float="center">
  <img src="https://upload.wikimedia.org/wikipedia/commons/e/e9/4_RGB_McCombs_School_Brand_Branded.png" width="300" height="100"/>
  <img src="https://mma.prnewswire.com/media/1458111/Great_Learning_Logo.jpg?p=facebook" width="200" height="100"/>
</p></center>

<center><font size=10>Artificial Intelligence and Machine Learning</center></font>
<center><font size=6>Natural Language Processing with Generative AI - Project</center></font>

<center><p float="center">
  <img src="https://i.ibb.co/Q325rK84/medical.png" width="480"/>
</p></center>

<center><font size=6>Medical Assistant</center></font>

## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

**1. Diagnostic Assistance**: "What are the common symptoms and treatments for pulmonary embolism?"

**2. Drug Information**: "Can you provide the trade names of medications used for treating hypertension?"

**3. Treatment Plans**: "What are the first-line options and alternatives for managing rheumatoid arthritis?"

**4. Specialty Knowledge**: "What are the diagnostic steps for suspected endocrine disorders?"

**5. Critical Care Protocols**: "What is the protocol for managing sepsis in a critical care unit?"

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## **Please read the instructions carefully before starting the project.**

This is a blank Python Notebook file in which only the section headers, task descriptions, and code comments are provided. Unlike the low-code version, no starter code or blanks ('_____') are given here.
* Read the comment(s) above each empty code cell carefully to understand the task, and then write the complete code yourself in that cell.
* Please run the codes in a sequential manner from the beginning to avoid any unnecessary errors.
* Add the results/observations (wherever mentioned) derived from the analysis in the presentation and submit the same.

**Note**: To run the LLM calls in this notebook, you will need an OpenAI-compatible API key (accessed here via the GL Proxy).

- Create a `config.json` file in the same directory as this notebook (e.g. `/content/config.json` on Google Colab) with the following structure:
```
{"OPENAI_API_KEY": "sk-...", "OPENAI_API_BASE": "https://.../v1"}
```
- Upload this `config.json` file to your working directory (e.g. via the Colab file browser) before running the "Setting up the OpenAI Client" cell below.
- Do **not** share your `config.json` file or commit it to any public repository, since it contains your private API key.

## Installing and Importing Necessary Libraries and Dependencies

In [ ]:
# For installing the required libraries
!pip install -q \
    pandas==2.2.2 \
    tiktoken==0.12.0 \
    pymupdf==1.26.5 \
    langchain==0.3.27 \
    langchain-community==0.3.31 \
    langchain-huggingface==0.3.1 \
    chromadb==1.1.1 \
    sentence-transformers \
    openai==2.50.0

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [ ]:
#Libraries for processing dataframes,text
import json,os
import tiktoken
import pandas as pd

#Libraries for Loading Data, Chunking, Embedding, and Vector Databases
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

## Question Answering using LLM

### Setting up the OpenAI Client

In [ ]:
# Load OpenAI credentials from config.json
# config.json is expected to be in the same directory as this notebook and look like:
# {"OPENAI_API_KEY": "sk-...", "OPENAI_API_BASE": "https://api.openai.com/v1"}
import json
from openai import OpenAI

with open("/content/config.json", "r") as f:
    config = json.load(f)

client = OpenAI(
    api_key=config["OPENAI_API_KEY"],
    base_url=config["OPENAI_API_BASE"]
)

In [ ]:
# Separate models are used for response generation and for LLM-as-a-judge evaluation.
# Using a different (and typically stronger) model as the judge avoids the generator
# grading its own homework, which is a known bias in LLM-as-a-judge setups.
llm = "gpt-4o-mini"    # used for generating answers (base LLM, prompt-engineered LLM, and RAG)
judge_llm = "gpt-4o"   # used only for the groundedness/relevance judge in the evaluation section

### Response

In [ ]:
def response(query, max_tokens=128, temperature=0, top_p=0.95):
    model_output = client.chat.completions.create(
        model=llm,
        messages=[{"role": "user", "content": query}],
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
    )

    return model_output.choices[0].message.content

In [ ]:
print(response("What treatment options are available for managing hypertension?"))

**Observations:**
- With no system prompt or retrieved context, `gpt-4o-mini` answers purely from its pretraining knowledge, so the response is a generic, textbook-style overview of hypertension management (lifestyle changes, diuretics, ACE inhibitors/ARBs, calcium channel blockers, beta-blockers) rather than anything sourced from the Merck Manual.
- Because it is an instruction-tuned chat model, the answer is already reasonably well organized (short paragraphs/bullets) even without explicit prompt engineering, unlike a raw base-model completion.
- At `temperature=0` and the default `max_tokens=128`, the answer is deterministic but may be cut off before covering every drug class.

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
query_1 = "What is the protocol for managing sepsis in a critical care unit?"
print(response(query_1))

**Observations:**
- Without any grounding context, `gpt-4o-mini` answers purely from its pretraining knowledge, so the response tends to be a generic, well-organized summary of sepsis management (early recognition, blood cultures, broad-spectrum antibiotics, fluids, vasopressors, source control) rather than the exact protocol/terminology used by the Merck Manual.
- The model may omit manual-specific details (e.g., precise drug names, dosing, or bundle timing) since it has no access to the source document.
- At the default `max_tokens=128`, the answer may be truncated before covering the full protocol.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
query_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
print(response(query_2))

**Observations:**
- Appendicitis and appendectomy are common medical knowledge, so the model is likely to correctly state that surgery is generally required and medicine alone is not curative.
- The answer may not reflect the manual's specific nuances (e.g., when antibiotics-first management is considered for uncomplicated cases), since there is no grounding context.
- The response reads as a generic overview rather than a manual-sourced clinical answer.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
query_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
print(response(query_3))

**Observations:**
- The query describes symptoms (patchy hair loss/bald spots) without naming the condition, so the model must first infer that this is likely alopecia areata before suggesting treatments - a step that can introduce inaccuracies without grounding.
- Expect a broad, somewhat generic list of causes and treatments rather than the manual's specific treatment hierarchy (e.g., corticosteroids, immunotherapy).
- This question is a good candidate for comparing against the RAG-based answer later, since descriptive (not diagnosis-named) queries are harder for a context-free LLM.

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
query_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
print(response(query_4))

**Observations:**
- This question (traumatic brain injury) has two parts - temporary vs. permanent impairment - and the default `max_tokens=128` may truncate the answer before both are addressed.
- Expect generic advice (rest, monitoring, surgery for severe cases) rather than the manual's severity-based classification and specific management protocols (e.g., ICP management for severe TBI).

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
query_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
print(response(query_5))

**Observations:**
- The model gives sensible general first-aid advice (immobilize, seek medical care) since this is common knowledge, but is unlikely to reflect the manual's specific fracture classification or orthopedic treatment/recovery guidance.
- The question has three parts (precautions, treatment, recovery/care considerations); with only 128 tokens the answer may address just one or two before being cut off.

## Question Answering using LLM with Prompt Engineering

In [ ]:
system_prompt = (
    "You are a highly knowledgeable medical assistant supporting healthcare professionals and patients. "
    "Answer the user's medical question accurately, concisely, and in a clear, well-organized format "
    "(use numbered steps, short headings, or bullet points wherever they improve readability). "
    "Base your answer on established, evidence-based medical knowledge and avoid speculation. "
    "If the question has multiple parts (e.g., causes, symptoms, treatment, and precautions), address each part explicitly. "
    "Always end with a brief note that this information supports, but does not replace, the judgement of a licensed healthcare professional."
)

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
prompt_1 = f"{system_prompt}\n\nQuestion: {query_1}"

# Combination 1: role/format-focused prompt engineering, larger token budget, deterministic decoding
print(response(prompt_1, max_tokens=256, temperature=0, top_p=0.95))

**Observations:**
- Compared to the plain query, the structured system prompt (role, formatting, "address each part") produces a more organized, protocol-like answer, with clearer separation between recognition, treatment steps, and escalation.
- Raising `max_tokens` to 256 (from the default 128) reduces the risk of the answer being cut off mid-list.
- `temperature=0` keeps the ordering and content deterministic across reruns.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
prompt_2 = f"{system_prompt}\n\nQuestion: {query_2}"

# Combination 2: same prompt-engineered instruction, slightly higher temperature/lower top_p
print(response(prompt_2, max_tokens=300, temperature=0.2, top_p=0.9))

**Observations:**
- The "address each part explicitly" instruction pushes the model to directly answer the embedded yes/no sub-question ("can it be cured via medicine?") instead of glossing over it, which was a risk in the unstructured version.
- A small temperature increase (0.2) adds minor lexical variety without compromising factual consistency.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
prompt_3 = f"{system_prompt}\n\nQuestion: {query_3}"

# Combination 3: moderate temperature/top_p, larger token budget for a descriptive, open-ended question
print(response(prompt_3, max_tokens=280, temperature=0.4, top_p=0.85))

**Observations:**
- The formatting instruction nudges the model to name the underlying condition (e.g., alopecia areata) before listing treatments, improving the logical structure of the answer versus the plain-query version.
- A moderate `temperature`/`top_p` (0.4/0.85) trades a bit of determinism for more natural phrasing, which suits an open-ended, descriptive question like this one.

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
prompt_4 = f"{system_prompt}\n\nQuestion: {query_4}"

# Combination 4: low temperature, larger token budget for a multi-part (temporary vs. permanent) answer
print(response(prompt_4, max_tokens=350, temperature=0.1, top_p=0.95))

**Observations:**
- The "address each part explicitly" instruction encourages the model to separate temporary vs. permanent impairment, and immediate vs. long-term treatment, producing a safer, more clinically organized response than the plain-query version.
- `temperature=0.1` keeps facts consistent while `max_tokens=350` gives enough room to cover both parts of this two-section answer without truncation.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
prompt_5 = f"{system_prompt}\n\nQuestion: {query_5}"

# Combination 5: moderate temperature/top_p, larger token budget for a three-part checklist-style answer
print(response(prompt_5, max_tokens=300, temperature=0.3, top_p=0.9))

**Observations:**
- The formatting instruction tends to produce an actionable, easy-to-follow checklist covering first aid through recovery, addressing the multi-part nature of the question better than a free-form response.
- Across all five combinations, the prompt-engineered instruction (role, formatting, "address each part") had a more visible effect on answer usability than the parameter changes alone, though a larger `max_tokens` was necessary whenever the question had multiple sections.

## Data Preparation for RAG

### Loading the Data

In [ ]:
# uncomment and run the below code snippets if the dataset is present in the Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
manual_pdf_path = "/content/medical_diagnosis_manual.pdf"

pdf_loader = PyMuPDFLoader(manual_pdf_path)
manual = pdf_loader.load()

### Data Overview

#### Checking the first 5 pages

In [ ]:
for i in range(5):
    print(f"Page Number : {i+1}", end="\n")
    print(manual[i].page_content, end="\n")
    print("-" * 100)

#### Checking the number of pages

In [ ]:
print(f"Total number of pages in the manual: {len(manual)}")

### Data Chunking

In [ ]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=1000,
    chunk_overlap=100
)

document_chunks = pdf_loader.load_and_split(text_splitter)

print(f"Total number of chunks created: {len(document_chunks)}")

for i in [0, 2, 3]:
    print(f"Chunk {i}:")
    print(document_chunks[i].page_content)
    print("-" * 100)

**Observations:**
- A `chunk_size` of 1000 tokens with a 100-token `chunk_overlap` keeps each chunk large enough to contain a coherent explanation (e.g., a full paragraph on symptoms or treatment) while still being small enough to fit comfortably inside the retrieval context window.
- The 100-token overlap means consecutive chunks share some text at their boundary, which reduces the chance that a sentence describing a treatment or dosage is split across two chunks and only partially retrieved.
- The total chunk count is much larger than the page count (4,000+ pages), since each page typically yields multiple ~1000-token chunks.

### Embedding

In [ ]:
embedding_model = HuggingFaceEmbeddings(model_name="BAAI/bge-base-en-v1.5")

embedding_1 = embedding_model.embed_query(document_chunks[0].page_content)
embedding_2 = embedding_model.embed_query(document_chunks[1].page_content)

print(f"Embedding dimension for chunk 0: {len(embedding_1)}")
print(f"Embedding dimension for chunk 1: {len(embedding_2)}")

assert len(embedding_1) == len(embedding_2), "Embedding dimensions should match across chunks"

### Vector Database

In [ ]:
out_dir = 'medical_db'

if not os.path.exists(out_dir):
  os.makedirs(out_dir)

In [ ]:
vectorstore = Chroma.from_documents(
    document_chunks,
    embedding_model,
    persist_directory=out_dir
)

# Reload the persisted vector store from disk
vectorstore = Chroma(
    persist_directory=out_dir,
    embedding_function=embedding_model
)

sample_query = "What treatment options are available for managing hypertension?"
similar_chunks = vectorstore.similarity_search(sample_query, k=3)

for i, chunk in enumerate(similar_chunks, start=1):
    print(f"Result {i}:")
    print(chunk.page_content)
    print("-" * 100)

**Observations:**
- The vector store persists the embeddings to `out_dir`, so it can be reloaded later (e.g., in a new session) without re-embedding the entire manual.
- The top-3 chunks returned for the hypertension query should be topically relevant (blood pressure, antihypertensive drug classes), confirming that the embedding model is placing semantically similar text close together in vector space.

### Retriever

In [ ]:
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 3}
)

retrieved_docs = retriever.invoke(sample_query)

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"Retrieved chunk {i}:")
    print(doc.page_content)
    print("-" * 100)

**Observations:**
- The retriever wraps the vector store with a fixed search configuration (`similarity` search, `k=3`), so downstream code can fetch relevant chunks without repeating the search parameters each time.
- `k=3` is a reasonable starting point - enough chunks to give the LLM some context without overwhelming it - and is one of the parameters we revisit in the fine-tuning section below.

In [ ]:
no_context_response = client.chat.completions.create(
    model=llm,
    messages=[{"role": "user", "content": "What treatment options are available for managing hypertension?"}],
    max_tokens=200,
    temperature=0,
    top_p=0.95,
)

print(no_context_response.choices[0].message.content)

**Observations:**
- Called directly through the OpenAI client without any retrieved context, the response is a generic, pretraining-based answer (similar in spirit to the plain LLM Q&A section) rather than one grounded in the Merck Manual.
- This serves as the "before" baseline that the RAG-based response (with retrieved context) is compared against next.

The above response is somewhat generic and is solely based on the data the model was trained on, rather than the medical manual.

Let's now provide our own context.

### System and User Prompt Template

Prompts guide the model to generate accurate responses. Here, we define two parts:

    1. The system message describing the assistant's role.
    2. A user message template including context and the question.

In [ ]:
qna_system_message = """
You are an assistant to a medical professional. Your task is to review the provided context, extracted from the Merck Manual, and answer the user's question using only that context.
User input will have the context required by you to answer user questions. This context will begin with the token: ###Context.
The context contains references to specific portions of the medical manual relevant to the user query.

User questions will begin with the token: ###Question.

Please answer only using the context provided in the input. Do not mention anything about the context in your final answer. Do not provide any information that is not backed by the given context.

If the answer is not found in the context, respond "I don't know, this is not covered in the provided medical manual." Do not try to make up an answer.

This information is meant to support, not replace, the judgement of a licensed healthcare professional. Do not provide medical advice beyond what is stated in the context.
"""

qna_user_message_template = """
###Context
Here are some documents that are relevant to the question mentioned below.
{context}

###Question
{question}
"""

### Response Function

In [ ]:
def generate_rag_response(user_input, k=3, max_tokens=128, temperature=0, top_p=0.95):
    global qna_system_message, qna_user_message_template

    # Retrieve the top-k relevant document chunks for the query
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input, k=k)
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine the retrieved chunks into a single context string
    context_for_query = ". ".join(context_list)

    # Fill the user message template with the context and the question
    user_message = qna_user_message_template.format(context=context_for_query, question=user_input)

    # Generate the response
    try:
        completion = client.chat.completions.create(
            model=llm,
            messages=[
                {"role": "system", "content": qna_system_message},
                {"role": "user", "content": user_message},
            ],
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
        )
        answer = completion.choices[0].message.content.strip()
    except Exception as e:
        answer = f'Sorry, I encountered the following error: \n {e}'

    return answer

## Question Answering using RAG

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
user_input_1 = query_1
print(generate_rag_response(user_input_1))

**Observations:**
- Grounded in the manual's sepsis chapter, the retrieved chunks should let the model cite the actual clinical bundle (recognition, cultures/labs, broad-spectrum antibiotics, fluids, vasopressors, source control) using manual-specific terminology, which is more precise than the no-context answer earlier.
- If the answer still looks generic, the retriever's `k=3` may not be pulling the exact page(s) discussing sepsis protocol - worth inspecting `retriever.invoke(user_input_1)` directly.
- The system prompt's "I don't know" fallback should prevent hallucinated details that aren't in the retrieved context.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
user_input_2 = query_2
print(generate_rag_response(user_input_2))

**Observations:**
- The retrieved context should confirm surgery (appendectomy) as the standard treatment and describe when non-operative/antibiotic management is considered, grounded directly in the manual rather than general knowledge.
- Because appendicitis is likely covered in one focused chapter, this is a good candidate for a high-groundedness answer, useful as a baseline when comparing against the evaluation scores later.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
user_input_3 = query_3
print(generate_rag_response(user_input_3))

**Observations:**
- Because the query describes symptoms rather than naming "alopecia areata" directly, retrieval quality depends heavily on semantic similarity between the description and the manual's wording.
- If the top-k chunks miss the relevant dermatology chapter, the RAG answer may default to "I don't know" per the system prompt - a good signal for whether `k` or the embedding model needs tuning (see the Fine-tuning section below).

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
user_input_4 = query_4
print(generate_rag_response(user_input_4))

**Observations:**
- Traumatic brain injury management is typically split across severity levels in the manual; watch whether the default `k=3` retrieves both the acute-management and rehabilitation portions, or only one.
- This is a strong motivating example for increasing `k` and `max_tokens`, tested explicitly in the fine-tuning step below.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
user_input_5 = query_5
print(generate_rag_response(user_input_5))

**Observations:**
- Fracture care often spans multiple manual sections (first aid, orthopedic management, rehabilitation); the response quality here is a good test case for increasing `k` to capture a broader set of relevant chunks.
- Compared to the no-context and prompt-engineered versions, the RAG answer should ground precautions/treatment/recovery guidance in the manual's actual orthopedic content rather than generic first-aid knowledge.

### Fine-tuning

#### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
# Combination 1: same k/max_tokens/top_p as baseline, higher temperature
print(generate_rag_response(user_input_1, temperature=0.5))

**Observations:**
- Raising `temperature` from 0 to 0.5 introduces more variety in phrasing, but since the answer is still constrained to the retrieved context (and the "don't know if not in context" instruction), the clinical content should stay largely the same - this parameter mainly affects wording, not facts.
- For a protocol-style answer like sepsis management, a lower temperature is generally preferable to keep the ordering of steps consistent across reruns.

#### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
# Combination 2: same k/max_tokens/temperature as baseline, lower top_p (more focused sampling)
print(generate_rag_response(user_input_2, top_p=0.7))

**Observations:**
- Lowering `top_p` to 0.7 restricts the model to a narrower set of high-probability tokens, which should make the answer slightly more concise and less likely to introduce tangential phrasing, at some cost to lexical variety.
- Since `temperature=0` by default, the effect of `top_p` alone is modest here.

#### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
# Combination 3: same k/temperature/top_p as baseline, larger max_tokens for a fuller answer
print(generate_rag_response(user_input_3, max_tokens=400))

**Observations:**
- Increasing `max_tokens` from 128 to 400 gives the model room to both name the likely condition and list multiple treatment options in full, rather than being cut off after the first one or two.
- This is especially useful for descriptive, open-ended questions like this one, where the retrieved context may span causes, diagnosis, and several treatment options.

#### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
# Combination 4: same k/max_tokens as baseline, moderate temperature and top_p together
print(generate_rag_response(user_input_4, temperature=0.3, top_p=0.85))

**Observations:**
- Combining a moderate `temperature` (0.3) with a slightly reduced `top_p` (0.85) balances natural phrasing against factual consistency, which suits a two-part clinical question (temporary vs. permanent impairment) where some rewording is fine but the core facts should stay grounded.
- The default `max_tokens=128` may still be tight for this multi-part question; combining this with a larger `max_tokens` (as tested in Combination 3) would likely help further.

#### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
# Combination 5: chunking + retriever variation - rebuild the corpus with a smaller chunk_size/overlap,
# use a larger k, and compare against the baseline retriever/chunking
alt_text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=500,
    chunk_overlap=50
)
alt_document_chunks = pdf_loader.load_and_split(alt_text_splitter)
print("Original chunking (1000/100) ->", len(document_chunks), "chunks")
print("Alternate chunking (500/50)  ->", len(alt_document_chunks), "chunks")

alt_out_dir = "medical_db_alt"
if not os.path.exists(alt_out_dir):
    os.makedirs(alt_out_dir)

alt_vectorstore = Chroma.from_documents(alt_document_chunks, embedding_model, persist_directory=alt_out_dir)
alt_retriever = alt_vectorstore.as_retriever(search_type='similarity', search_kwargs={'k': 6})

# Temporarily swap the global retriever used inside generate_rag_response() to the alternate one
original_retriever = retriever
retriever = alt_retriever

print(generate_rag_response(user_input_5, k=6, max_tokens=300, temperature=0.15, top_p=0.92))

retriever = original_retriever  # restore the default (baseline) retriever used elsewhere in the notebook

**Observations:**
- **Chunking:** smaller chunks (500/50) roughly double the number of chunks versus the baseline (1000/100). Smaller chunks can improve retrieval precision (less irrelevant text per chunk) but risk splitting a single explanation across chunks, which is why `chunk_overlap` matters.
- **Retriever (`k`):** a larger `k=6` on the alternate (smaller-chunk) index pulls in more, narrower chunks, which should help this multi-part question (precautions, treatment, recovery) draw on multiple manual sections rather than just one.
- **LLM parameters:** the moderate `temperature`/`top_p` (0.15/0.92) slightly diversifies phrasing without introducing noticeable hallucination, since the answer stays constrained by the retrieved context.
- Overall, across Combinations 1-5, `k` and chunk size have a larger impact on answer completeness than the LLM sampling parameters (`temperature`, `top_p`) alone for this grounded RAG setup.

## Output Evaluation

Let us now use the LLM-as-a-judge method to check the quality of the RAG system on two parameters - retrieval and generation. We illustrate this evaluation based on the answers generated to the questions from the previous section.

- We use a **different model as the judge** (`judge_llm`) from the one used for answer generation (`llm`), via the GL Proxy. This avoids the generator model rating its own output, which can otherwise inflate scores due to self-preference bias.

In [ ]:
groundedness_rater_system_message = """
You are tasked with rating AI-generated answers to questions posed by users about a medical manual.
You will be presented a question, context used by the AI system to generate the answer, and an AI-generated answer to the question.
In the input, the question will begin with ###Question, the context will begin with ###Context, while the AI-generated answer will begin with ###Answer.

Evaluation criteria:
The task is to judge the extent to which the metric is followed by the answer.
1 - The metric is not followed at all
2 - The metric is followed only to a limited extent
3 - The metric is followed to a good extent
4 - The metric is followed mostly
5 - The metric is followed completely

Metric:
The answer should be derived only from the information presented in the context. It should not contain any information that is not supported by, or that contradicts, the given context.

Instructions:
- Briefly check whether every claim made in the answer is supported by the context provided.
- Do not output any explanation, reasoning, or extra text.
- Return only the final score, as a dictionary in the exact format: {"groundedness_score": <score>}
"""

In [ ]:
relevance_rater_system_message = """
You are tasked with rating AI-generated answers to questions posed by users about a medical manual.
You will be presented a question, context used by the AI system to generate the answer, and an AI-generated answer to the question.
In the input, the question will begin with ###Question, the context will begin with ###Context, while the AI-generated answer will begin with ###Answer.

Evaluation criteria:
The task is to judge the extent to which the metric is followed by the answer.
1 - The metric is not followed at all
2 - The metric is followed only to a limited extent
3 - The metric is followed to a good extent
4 - The metric is followed mostly
5 - The metric is followed completely

Metric:
Relevance measures how well the answer addresses the main aspects of the question, based on the context provided.

Instructions:
- Briefly check whether the answer addresses every main aspect of the question, using the context provided.
- Do not output any explanation, reasoning, or extra text.
- Return only the final score, as a dictionary in the exact format: {"relevance_score": <score>}
"""

In [ ]:
user_message_template = """
###Question
{question}

###Context
{context}

###Answer
{answer}
"""

In [ ]:
def generate_ground_relevance_response(user_input, k=3, max_tokens=128, temperature=0, top_p=0.95):
    global qna_system_message, qna_user_message_template

    # Retrieve the relevant document chunks and combine them into a context, as in generate_rag_response()
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input, k=k)
    context_list = [d.page_content for d in relevant_document_chunks]
    context_for_query = ". ".join(context_list)

    # Generate an answer using the GENERATION model (llm)
    user_message = qna_user_message_template.format(context=context_for_query, question=user_input)
    answer_completion = client.chat.completions.create(
        model=llm,
        messages=[
            {"role": "system", "content": qna_system_message},
            {"role": "user", "content": user_message},
        ],
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
    )
    answer = answer_completion.choices[0].message.content.strip()

    # Fill the evaluation template with the question, context, and generated answer
    eval_user_message = user_message_template.format(
        context=context_for_query,
        question=user_input,
        answer=answer,
    )

    # Ask the JUDGE model (judge_llm) to rate groundedness
    groundedness_response = client.chat.completions.create(
        model=judge_llm,
        messages=[
            {"role": "system", "content": groundedness_rater_system_message},
            {"role": "user", "content": eval_user_message},
        ],
        max_tokens=max_tokens,
        temperature=0,
    )

    # Ask the JUDGE model (judge_llm) to rate relevance
    relevance_response = client.chat.completions.create(
        model=judge_llm,
        messages=[
            {"role": "system", "content": relevance_rater_system_message},
            {"role": "user", "content": eval_user_message},
        ],
        max_tokens=max_tokens,
        temperature=0,
    )

    return groundedness_response.choices[0].message.content.strip(), relevance_response.choices[0].message.content.strip()

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
groundedness_1, relevance_1 = generate_ground_relevance_response(user_input_1, max_tokens=370)

print(groundedness_1, end="\n\n")
print(relevance_1)

**Observations:**
- Expect a high groundedness score if the retrieved sepsis-protocol context was on-topic (as seen in the earlier RAG answer); the relevance score should likewise be high since the question is precise and factual, giving the retriever a strong signal.
- If either score is low, check whether the retrieved chunks actually discuss sepsis management, or whether the answer drifted into unsupported claims.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
groundedness_2, relevance_2 = generate_ground_relevance_response(user_input_2, max_tokens=350)

print(groundedness_2, end="\n\n")
print(relevance_2)

**Observations:**
- Since appendicitis is well-covered in a single, focused manual chapter, expect both groundedness and relevance to score highly, similar to the pattern seen for narrowly-scoped, well-retrieved questions.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
groundedness_3, relevance_3 = generate_ground_relevance_response(user_input_3, max_tokens=350)

print(groundedness_3, end="\n\n")
print(relevance_3)

**Observations:**
- This is the best candidate to reveal a groundedness/relevance gap - if the retriever pulled loosely related dermatology content (since the query never names "alopecia areata"), the LLM-judge should assign a lower score with an explanation flagging missing or partially-supported information.

### Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
groundedness_4, relevance_4 = generate_ground_relevance_response(user_input_4, max_tokens=500)

print(groundedness_4, end="\n\n")
print(relevance_4)

**Observations:**
- This multi-part question (treatment for temporary vs. permanent impairment) is a good test of relevance - a lower score here would indicate the RAG answer addressed only part of the question, reinforcing the case for the higher `k`/`max_tokens` tested in the fine-tuning step.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
groundedness_5, relevance_5 = generate_ground_relevance_response(user_input_5, max_tokens=500)

print(groundedness_5, end="\n\n")
print(relevance_5)

**Observations:**
- Similar to Q4, this question spans first aid, treatment, and recovery; watch whether the relevance judge penalizes the answer for covering only one aspect (e.g., first aid but not recovery precautions) - a useful signal for whether `k` should be increased for broad, multi-topic questions.
- Across all five evaluations, groundedness and relevance scores together give a scalable way to spot which questions need retrieval or prompt tuning, without a clinician manually reviewing every answer.

## Actionable Insights and Business Recommendations

**Instructions**: Based on the responses observed across the base LLM, prompt-engineered LLM, RAG, fine-tuning, and evaluation sections above, write down your actionable insights and business recommendations below. Consider aspects such as response completeness (`max_tokens`), groundedness and relevance scores, the effect of retrieval parameters (`k`, chunk size/overlap), and how this solution could be scaled or improved for real-world use.

1. A RAG pipeline grounded in the Merck Manual measurably improves answer quality over the bare `gpt-4o-mini` LLM: retrieved context lets the model cite manual-specific protocols, drug names, and procedures instead of falling back on generic pretraining knowledge, directly reducing the risk of hallucinated or outdated clinical guidance.

2. Prompt engineering (role-setting, explicit output structure, "address each part explicitly") meaningfully improves answer usability even without retrieval, and remains useful on top of RAG to keep answers well-organized (e.g., symptoms vs. treatment, emergency vs. long-term care).

3. Retrieval quality is the main lever for answer quality: broader, multi-topic questions (e.g., brain injury, leg fracture) benefit from a higher `k` and/or smaller chunk size, while narrow factual questions (e.g., appendicitis) do well with a smaller, more focused context. `chunk_size`/`chunk_overlap` should be tuned together with `k`, since they jointly determine how much of the source manual is visible to the model for any given answer.

4. The "I don't know" instruction embedded in the system prompt, together with a separate judge model (`judge_llm`) for evaluation, is critical for a medical use case: it constrains the assistant to the manual's content, avoids fabricating clinical advice when the manual doesn't cover a topic, and avoids the generator model rating its own output (self-preference bias).

5. For deployment, the assistant should be positioned as a decision-support tool (not a diagnostic authority) for clinicians and triage staff, with groundedness/relevance scores tracked over time as a quality gate whenever the manual, chunking strategy, or model is updated - and a human-in-the-loop review step added for high-stakes queries before production use.

<font size=6 color='blue'>Power Ahead</font>
___